In [1]:
import pandas as pd
import sys

# sys.path.append('../input/baseline-infer/Kaggle_kernel_infer/features')
sys.path.insert(2, '../input/baseline-infer/Kaggle_kernel_infer/Kaggle_kernel_infer/features')
sys.path.insert(1, '../input/configs')

import os
import datetime
import copy

# from data.dataloader.build import create_dataloader
# from solver.optimizer import create_optimizer
# from solver.lr_scheduler import wrapper_lr_scheduler
# from engine.trainer import *
# from data.dataloader import create_dataloader

from utils.utils import *
from model.build import build_model
from configs import load_args, merge_from_arg

from torch.utils.data import Dataset, DataLoader
import torch
import cv2

from data.transforms.build import build_transforms

ModuleNotFoundError: No module named 'utils'

In [2]:
# LOAD MODEL CONFIGS (DO NOT EXEC THE SECOND TIME)
from resnet50_flip_rorate_crop_ssr_iaa_gm_cj_re_rp_pretrained_640 import config as cfg

cfg['model']['backbone']['pretrained'] = False
weights_path = '../input/wwwwww/weight_epoch_17.pth' # 改成你刚上传的文件地址

ModuleNotFoundError: No module named 'resnet50_flip_rorate_crop_ssr_iaa_gm_cj_re_rp_pretrained_640'

In [3]:
# 小修小改
cfg['val_pipeline']['transforms'][0]['output_size'] = 640
cfg['val_pipeline']['transforms'][0]

NameError: name 'cfg' is not defined

In [4]:
import model.net as net

cfg_c = copy.deepcopy(cfg)
if 'net' in cfg["model"].keys():
    net_cfg = cfg['model']['net']
    net_type = net_cfg.pop('type')
    model = getattr(net, net_type)(cfg_c)
else:
    raise KeyError("net{} is not found!!!".format(cfg))

model_state_dict = model.state_dict()
state_dict = torch.load(weights_path, map_location='cpu')

if 'model' in state_dict.keys():
    state_dict = state_dict['model']
for key in state_dict.keys():
    if key in model_state_dict.keys() and state_dict[key].shape == model_state_dict[key].shape:
        model_state_dict[key] = state_dict[key]

assert len(state_dict.keys()) == len(model_state_dict.keys())
model.load_state_dict(model_state_dict)


ModuleNotFoundError: No module named 'model'

In [5]:
import random
class Infer_set(Dataset):
    def __init__(self, path, transforms, device):
        super(Infer_set, self).__init__()
        import os
        import cv2
        self.path = path
        self.device = device
        self.transforms = transforms
        self.img_list = [img for img in os.listdir(path) if img.endswith('.jpg')]
#         random.shuffle(self.img_list)
#         print(self.img_list)
        
        
    def __getitem__(self, idx):
        img = cv2.imread(os.path.join(self.path, self.img_list[idx]))
        if self.transforms is not None:
            img = self.transforms(img)
        return img.to(self.device), self.img_list[idx]
        
        
        
    def __len__(self):
        return len(self.img_list)
    

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
    
transforms = build_transforms(cfg['val_pipeline']['transforms'])
infer_set = Infer_set(path='../input/cassava-leaf-disease-classification/test_images', transforms=transforms, device=device)
infer_loader = DataLoader(infer_set, batch_size=32, shuffle=False)
infer_loader.__iter__().__next__()[0].shape
# infer_loader.batch_size

NameError: name 'Dataset' is not defined

In [6]:
# import matplotlib.pyplot as plt
# plt.imshow(infer_set[144][0].cpu().permute(1,2,0).numpy()[:,:,::-1])
transforms

NameError: name 'transforms' is not defined

In [7]:
# INFER
f = open('./submission.csv', 'w')
f.write('image_id,label\n')
idx = 0

with torch.no_grad():
    model.eval()
    for i, (image, fns) in enumerate(infer_loader):
#         print(image.shape)
        preds = model(image)
        preds = preds.argmax(1).squeeze().cpu().numpy()
#         print(preds)

        if len(fns)>1:
            for (pred, fn) in zip(preds, fns):
                f.write(fn + ',' + str(pred) + '\n')
                idx += 1
        else:
            f.write(fns[0] + ',' + str(preds) + '\n')
            idx += 1
f.close()

NameError: name 'torch' is not defined

In [8]:
f = open('./submission.csv', 'r')

print(f.readlines())

f.close()

['image_id,label\n']
